In [ ]:
import os
import sys
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
from xgboost import XGBClassifier
from sklearn.svm import SVC
from tqdm.notebook import tqdm


In [ ]:
path="C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Paper/RQ3/Classification/MiddleWindow"

## Function to Perform Classification

In [ ]:
def get_classification(classifier, classifier_initialisation, cv_methods, cv_strategies, X, y_encoded, participants, focus, num_class):

    if num_class == 2:
        df_clf_result= pd.DataFrame(columns=['Classifier', 'CV', 'Accuracy', 
                                             'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                             'Best Group', 'Best Accuracy','Best Confusion Matrix' ])
    elif num_class == 3:
        df_clf_result= pd.DataFrame(columns=['Classifier', 'CV', 'Accuracy', 
                                             'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                             'Best Group', 'Best Accuracy','Best Confusion Matrix' ])
    # Loop through classifiers and perform classification
    for classifier, clf in zip(classifier, classifier_initialisation):

        print(f"-----------------------Initializing  {classifier}----------------------- \n")        
        
        # Loop through methods and perform cross-validation
        for cv_method, cv_strategy in zip(cv_methods, cv_strategies):
            
            cv = cv_strategy
            print(f'*****Performing evaluation with {cv_method}*****\n' )
            accuracy_scores = []
            conf_matrices = []
            class_report = {}

            #Store the highest accuracy and confusion matrix to find best group
            max_accuracy = 0
            best_conf_matrix = 0
            best_group = None

            #assign group of focus for cross validation
            group = focus if cv_method in ['Leave-One-Algorithm-Out CV', 'Leave-One-Electrode-Out CV'] else participants
            

            if cv_method == 'Leave-One-Algorithm-Out CV':
                group_name = 'Algorithm'
            elif cv_method == 'Leave-One-Electrode-Out CV':
                group_name = 'Electrode'   
            elif cv_method == 'Leave-One-Subject-Out CV':
                group_name = 'Participant'
            else:
                group_name = 'None' 

            for fold,(train_index, test_index) in enumerate(cv.split(X, y_encoded, groups= group)):
                train_X, test_X = X.iloc[train_index], X.iloc[test_index]
                train_y, test_y = y_encoded[train_index], y_encoded[test_index]
                
                # algorithm chosen for testing in this fold
                test_group = group.iloc[test_index].unique()

                # Initialize and train Random Forest classifier
                clf.fit(train_X, train_y)

                # Make predictions on the test set
                predictions = clf.predict(test_X)

                # Accuracy
                accuracy = np.round((accuracy_score(test_y, predictions))*100, decimals=4)
                accuracy_scores.append(accuracy)

                # Confusion matrix
                conf_matrix = confusion_matrix(test_y, predictions)
                conf_matrices.append(conf_matrix)
                
                #Classification report 
                classification_rep = classification_report (test_y, predictions, output_dict=True)
                
                #Check for highest accuracy
                if accuracy > max_accuracy:
                    max_accuracy = accuracy
                    best_group = test_group
                    best_conf_matrix = conf_matrix

                # Print accuracy for each fold
                print(f'{fold + 1}. Testing {group_name} : {test_group} , Accuracy: {accuracy}, confusion matrix: {conf_matrix}')

                # Print classification report
                print(f"Classification Report:\n")
                for label, metrics in classification_rep.items():
                    if label != 'accuracy' and label != 'weighted avg' and label != 'macro avg' and label != 'micro avg':

                        if label not in class_report:
                            class_report[label] = {'precision': [], 'recall': [], 'f1-score': [], 'support': []}
                        
                        print(f"Class {label}:")
                        print(f"  Precision: {metrics['precision']:.2f}")
                        print(f"  Recall: {metrics['recall']:.2f}")
                        print(f"  F1 Score: {metrics['f1-score']:.2f}")
                        print(f"  Support: {metrics['support']}")
                        print()

                        class_report[label]['precision'].append(metrics['precision'])
                        class_report[label]['recall'].append(metrics['recall'])
                        class_report[label]['f1-score'].append(metrics['f1-score'])
                        class_report[label]['support'].append(metrics['support'])
                print("\n")
                
            #Average Accuracy over all folds
            average_accuracy = np.round((sum(accuracy_scores) / len(accuracy_scores)), decimals=4)
            print(f"\n Average Accuracy: {average_accuracy}")

            # Pad smaller matrices with zeros before summing to avoid shape mismatch in multiclass classification
            max_shape = max(cm.shape for cm in conf_matrices)
            conf_matrices_padded = [np.pad(cm, ((0, max_shape[0] - cm.shape[0]), (0, max_shape[1] - cm.shape[1])), 'constant') for cm in conf_matrices]

            #Average Confusion Matrix over all folds
            average_conf_matrix = sum(conf_matrices_padded) / len(conf_matrices_padded)
            print(f"\n Average Confusion Matrix: {np.round(average_conf_matrix, decimals=0)} \n")

            #Store the metrics in percentage
            average_class_reports = {}
            for label, metrics in class_report.items():
                average_class_reports[label] = {
                    'precision': np.round((np.mean(metrics['precision'])*100), decimals=2),
                    'recall': np.round((np.mean(metrics['recall'])*100), decimals=2),
                    'f1-score': np.round((np.mean(metrics['f1-score'])*100), decimals=2),
                    'support': np.round((np.mean(metrics['support'])*100), decimals=2)
                }
            # Print the average classification report for each class
            print("Average Classification Report across all folds for each class:")
            for label, metrics in average_class_reports.items():
                print(f"Class {label}:")
                print(f"  Precision: {metrics['precision']:.2f}")
                print(f"  Recall: {metrics['recall']:.2f}")
                print(f"  F1 Score: {metrics['f1-score']:.2f}")
                print(f"  Support: {metrics['support']:.2f}")
                print()

            if cv_method in ['Leave-One-Algorithm-Out CV', 'Leave-One-Electrode-Out CV', 'Leave-One-Subject-Out CV']:
                print(f'\n Maximum Accuracy --- Testing {group_name} : {best_group} , Accuracy: {max_accuracy}, confusion matrix : {best_conf_matrix} \n')
            
            else:
                group_name = None
                best_group = None
                max_accuracy = None
                best_conf_matrix = None
            #Save the result into a dataframe
            if num_class == 2:
                df_clf_result.loc[len(df_clf_result)]= [classifier, cv_method, average_accuracy, 
                                                        average_class_reports['0']['precision'], average_class_reports['0']['recall'], average_class_reports['0']['f1-score'], average_class_reports['0']['support'],
                                                        average_class_reports['1']['precision'], average_class_reports['1']['recall'], average_class_reports['1']['f1-score'], average_class_reports['1']['support'],
                                                        best_group, max_accuracy, best_conf_matrix]
            elif num_class == 3:
                df_clf_result.loc[len(df_clf_result)]= [classifier, cv_method, average_accuracy, 
                                                        average_class_reports['0']['precision'], average_class_reports['0']['recall'], average_class_reports['0']['f1-score'], average_class_reports['0']['support'],
                                                        average_class_reports['1']['precision'], average_class_reports['1']['recall'], average_class_reports['1']['f1-score'], average_class_reports['1']['support'],
                                                        average_class_reports['2']['precision'], average_class_reports['2']['recall'], average_class_reports['2']['f1-score'], average_class_reports['2']['support'],
                                                        best_group, max_accuracy, best_conf_matrix]
                

            
        
    return df_clf_result

# Data Processed with Common Average Referencing 

In [ ]:
# create folder to store results if not exist

result_path= path + "/CAR"
if not os.path.exists(result_path):
    os.makedirs(result_path)

In [ ]:
df_processed = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Paper/Processing/CAR/MiddleWindow/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")

df_processed

## Binary Classification

In [ ]:
# Log the outputs

log_file = open(result_path+'/binary_classification_output.log', 'w')
sys.stdout = log_file

In [14]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path+'/bin_clf_result.csv')
df_result
    

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are 

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,Expert F1-Score,Expert Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Stratified10-fold CV,85.1370,86.45,87.26,86.78,4080.00,83.76,82.44,82.98,3190.00,None,NaN
1,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Subject-Out CV,72.6083,60.00,53.91,56.64,2040.00,47.83,32.05,35.11,1386.96,[7],100.0
2,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Algorithm-Out CV,85.4017,87.16,87.26,87.02,1275.00,83.71,83.14,83.09,996.88,[BogoSort],96.0
3,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",SVM,Stratified10-fold CV,83.7614,80.76,93.62,86.67,4080.00,89.69,71.15,79.17,3190.00,None,NaN
4,Algorithm,Baseline,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Subject-Out CV,83.7500,75.00,74.61,74.80,2550.00,64.29,64.29,64.29,2278.57,[71],100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",SVM,Leave-One-Subject-Out CV,84.8750,75.00,72.56,73.65,5200.00,68.75,60.06,61.10,4800.00,[71],100.0
68,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",SVM,Leave-One-Electrode-Out CV,86.2500,84.15,90.75,87.28,1300.00,89.16,81.38,85.02,1200.00,[TP8],88.0
69,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",XGB,Stratified10-fold CV,92.2500,90.97,94.60,92.69,8320.00,94.05,89.70,91.74,7680.00,None,NaN
70,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(1600, 4)","(1600,)",XGB,Leave-One-Subject-Out CV,85.5000,63.16,59.38,61.14,4378.95,52.17,43.89,45.66,3339.13,[4],100.0


In [15]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

## Multiclass Classification

In [16]:
# Log the outputs

log_file = open(result_path+'/multiclass_classification_output.log', 'w')
sys.stdout = log_file

In [17]:
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path+'/mcl_clf_result.csv')
df_mcl_result
    

  0%|          | 0/8 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score a

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Stratified10-fold CV,65.3063,68.37,68.91,...,57.99,57.43,57.47,3450.00,70.56,69.29,69.70,3190.00,None,NaN
1,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Subject-Out CV,52.5805,42.86,29.13,...,32.26,14.82,18.79,1112.90,29.41,19.72,21.80,938.24,[24],100.0000
2,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Algorithm-Out CV,65.2299,67.58,68.31,...,59.68,58.65,58.43,1078.12,70.49,67.89,68.64,996.88,[GreatestCommonDivisor],77.4194
3,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",SVM,Stratified10-fold CV,63.2451,55.66,93.37,...,75.30,28.40,40.83,3450.00,79.24,62.43,69.50,3190.00,None,NaN
4,Algorithm,Baseline,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Subject-Out CV,54.4730,44.44,44.10,...,20.00,1.66,2.92,2300.00,44.44,44.44,44.44,1772.22,[71],100.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Subject-Out CV,58.3615,50.00,47.46,...,34.78,17.60,20.56,3339.13,52.63,32.40,35.19,4042.11,[71],100.0000
68,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Electrode-Out CV,66.9342,63.09,89.06,...,67.07,50.00,56.96,1200.00,75.34,59.90,66.43,1200.00,[C6],75.6757
69,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Stratified10-fold CV,79.0964,72.49,81.01,...,78.83,71.75,75.05,7680.00,88.07,84.37,86.09,7680.00,None,NaN
70,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Leave-One-Subject-Out CV,55.1098,52.17,39.61,...,33.33,14.77,17.76,2327.27,37.50,20.02,21.87,2400.00,[24],100.0000


In [18]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Data Processed with Baseline Correction Algorithm

In [19]:

# create folder to store results if not exist
result_path_bca= path + "/BCA"
if not os.path.exists(result_path_bca):
    os.makedirs(result_path_bca)

df_processed_bca = pd.read_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_Result/Paper/Processing/BCA/MiddleWindow/processed_data.csv").drop("Unnamed: 0", axis=1, errors="ignore")
df_processed = df_processed_bca.copy()
df_processed

,Focus,Data,FrequencyBand,FilePath
0,Algorithm,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
1,Algorithm,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
2,ElectrodePosition,CodeComprehension,4to50hz,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...
3,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,C:/Users/Mahima Acharya/Documents/BCI/Data/EEG...


## Binary classification

In [20]:
# Log the outputs

log_file = open(result_path_bca+'/binary_classification_output.log', 'a')
sys.stdout = log_file

In [21]:


# Define the number of classes for classification
num_class = 2
df_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier','CV','Accuracy', 
                                  'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                  'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                  'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
       
    # Binary Classification
    df_brain_waves = df_brain_waves[df_brain_waves['SkillLevel'].isin(['Novice', 'Expert'])]
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(random_state=42)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies, X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']

        # Add a new row to df_result
        df_result.loc[len(df_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy, 
                                         expert_precision, expert_recall, expert_f1_score, expert_support,
                                         novice_precision, novice_recall, novice_f1_score, novice_support,
                                          best_group, max_accuracy]


df_result.to_csv(result_path_bca+'/bin_clf_result.csv')
df_result
    

  0%|          | 0/4 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score a

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,Expert F1-Score,Expert Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Stratified10-fold CV,85.9475,88.08,86.73,87.29,4080.00,83.80,84.92,84.22,3190.00,None,NaN
1,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Subject-Out CV,76.6333,63.16,57.24,59.99,2147.37,41.67,34.51,36.61,1329.17,[38],100.0000
2,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Algorithm-Out CV,86.4624,88.15,87.84,87.75,1275.00,85.40,84.56,84.59,996.88,[HeightOfTree],95.8333
3,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Stratified10-fold CV,83.8984,80.81,93.87,86.80,4080.00,89.99,71.15,79.29,3190.00,None,NaN
4,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Subject-Out CV,83.7500,75.00,74.61,74.80,2550.00,64.29,64.29,64.29,2278.57,[71],100.0000
5,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Algorithm-Out CV,83.8717,80.67,93.99,86.77,1275.00,89.84,70.68,79.00,996.88,[ArrayAverage],87.5000
6,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Stratified10-fold CV,89.9448,90.02,92.38,91.09,4080.00,90.36,86.82,88.41,3190.00,None,NaN
7,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Leave-One-Subject-Out CV,86.5000,70.59,67.46,68.93,2400.00,57.14,48.36,50.20,1519.05,[71],100.0000
8,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",XGB,Leave-One-Algorithm-Out CV,90.9409,91.60,92.53,91.95,1275.00,90.44,88.82,89.45,996.88,[ArrayAverage],95.8333
9,Algorithm,CodeComprehension,AlphaBetaThetaGamma,"(727, 4)","(727,)",RF,Stratified10-fold CV,91.8779,90.96,95.11,92.94,4080.00,93.49,87.74,90.42,3190.00,None,NaN


In [22]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

## Multi-class Classification

In [23]:
# Log the outputs

log_file = open(result_path_bca+'/multiclass_classification_output.log', 'a')
sys.stdout = log_file

In [24]:
# Define the number of classes for classification
num_class = 3

df_mcl_result = pd.DataFrame(columns=['Focus', 'Data', 'FrequencyBand','X_shape','y_shape','Classifier', 'CV','Accuracy', 
                                      'Expert Precision','Expert Recall', 'Expert F1-Score', 'Expert Support',
                                             'Intermediate Precision','Intermediate Recall', 'Intermediate F1-Score', 'Intermediate Support', 
                                             'Novice Precision','Novice Recall', 'Novice F1-Score', 'Novice Support',  
                                      'Best Group', 'Best Accuracy'])


for index, row in tqdm(df_processed.iterrows(), total=len(df_processed)):
    file_path = row["FilePath"]
    focus= row["Focus"]
    data = row["Data"]
    freq_band= row["FrequencyBand"]
    df_brain_waves = pd.read_csv(file_path)
    print(f'Reading {file_path}')
    
    
    # Separate features and target variable
    if freq_band == '4to50hz':
        X = df_brain_waves[["Entropy"]]
    else:
        X = df_brain_waves[["Alpha_Entropy", "Beta_Entropy", "Gamma_Entropy", "Theta_Entropy"]]
        
    y = df_brain_waves['SkillLevel']
    participants = df_brain_waves['Participant']
    
    #Encoding the SkillLevel

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # The mapping between original labels and encoded values
    label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    # Print the mapping
    print("Label Encoding Mapping:")
    for label, encoded_value in label_mapping.items():
        print(f"{label} -> {encoded_value}")
    
    X_shape= X.shape
    y_shape= y_encoded.shape
   

    # Define classifiers and corresponding initialisations
    classifiers= ['RF','SVM','XGB']
    classifier_initialisations = [ RandomForestClassifier(n_estimators= 100,random_state=42),
                            SVC(kernel='linear', C=1, random_state=42),
                            XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=4)]
    #For multiclass, XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y)), random_state=42)
    # Define methods and corresponding cross-validation strategies
    if focus == 'Algorithm':
        algorithms = df_brain_waves["Algorithm"]        
        cv_methods_algorithm = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Algorithm-Out CV']
        cv_strategies_algorithm = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                GroupKFold(n_splits=len(participants.unique())),
                                GroupKFold(n_splits=len(algorithms.unique()))]
        cv_methods = cv_methods_algorithm
        cv_strategies = cv_strategies_algorithm
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                           X, y_encoded, participants, algorithms, num_class)
    else:
        electrodes = df_brain_waves["Channel"]        
        cv_methods_electrode = ['Stratified10-fold CV', 'Leave-One-Subject-Out CV', 'Leave-One-Electrode-Out CV']
        cv_strategies_electrode = [StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
                                        GroupKFold(n_splits=len(participants.unique())),
                                        GroupKFold(n_splits=len(electrodes.unique()))]

        cv_methods = cv_methods_electrode
        cv_strategies = cv_strategies_electrode    
        df_clf_result = get_classification(classifiers, classifier_initialisations, cv_methods, cv_strategies,
                                            X, y_encoded, participants, electrodes, num_class)


    # Iterate through each row in df_clf_result
    for index, row in df_clf_result.iterrows():
        # Extract relevant information from df_clf_result
        classifier = row['Classifier']
        cv = row['CV']
        accuracy = row['Accuracy']
        expert_precision = row['Expert Precision']
        expert_recall = row['Expert Recall']
        expert_f1_score = row['Expert F1-Score']
        expert_support = row['Expert Support']
        intermediate_precision = row['Intermediate Precision']
        intermediate_recall = row['Intermediate Recall']
        intermediate_f1_score = row['Intermediate F1-Score']
        intermediate_support = row['Intermediate Support']
        novice_precision = row['Novice Precision']
        novice_recall = row['Novice Recall']
        novice_f1_score = row['Novice F1-Score']
        novice_support = row['Novice Support']
        best_group = row ['Best Group']
        max_accuracy = row['Best Accuracy']
        # Add a new row to df_result
        df_mcl_result.loc[len(df_mcl_result)] = [focus, data, freq_band, X_shape, y_shape, classifier, cv, accuracy,  
                                                 expert_precision, expert_recall, expert_f1_score, expert_support,
                                                intermediate_precision, intermediate_recall, intermediate_f1_score, intermediate_support,
                                                novice_precision, novice_recall, novice_f1_score, novice_support,
                                                best_group, max_accuracy]
        
df_mcl_result.to_csv(result_path_bca+'/mcl_clf_result.csv')
df_mcl_result
    

  0%|          | 0/4 [00:00<?, ?it/s]

c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Python39\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score a

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support,Novice Precision,Novice Recall,Novice F1-Score,Novice Support,Best Group,Best Accuracy
0,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Stratified10-fold CV,69.0196,70.97,70.57,...,64.67,64.65,64.55,3450.00,71.48,71.80,71.52,3190.00,None,NaN
1,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Subject-Out CV,57.2160,48.00,33.50,...,35.48,19.64,23.24,1112.90,32.26,21.63,24.06,1029.03,[6],100.0000
2,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",RF,Leave-One-Algorithm-Out CV,68.9094,70.82,69.61,...,64.31,64.57,63.87,1078.12,74.28,72.29,72.35,996.88,[SiebDesEratosthenes],84.8485
3,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Stratified10-fold CV,66.0497,57.57,93.62,...,80.62,36.83,49.85,3450.00,80.18,62.43,69.87,3190.00,None,NaN
4,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Subject-Out CV,59.4769,44.44,44.21,...,33.33,13.79,16.82,2300.00,47.06,47.06,47.06,1876.47,[71],100.0000
5,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",SVM,Leave-One-Algorithm-Out CV,69.1861,60.46,93.75,...,84.69,46.51,59.78,1078.12,79.90,61.87,69.48,996.88,[SignChecker],72.7273
6,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Stratified10-fold CV,73.5073,71.03,82.85,...,71.96,56.85,63.05,3450.00,79.64,79.66,79.36,3190.00,None,NaN
7,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Leave-One-Subject-Out CV,57.0664,57.14,47.77,...,34.62,20.19,22.14,1326.92,36.00,23.33,26.06,1276.00,[71],100.0000
8,Algorithm,CodeComprehension,4to50hz,"(1072, 1)","(1072,)",XGB,Leave-One-Algorithm-Out CV,75.2458,71.88,85.22,...,77.03,60.72,67.18,1078.12,80.36,77.95,78.67,996.88,[SiebDesEratosthenes],87.8788
9,Algorithm,CodeComprehension,AlphaBetaThetaGamma,"(1072, 4)","(1072,)",RF,Stratified10-fold CV,79.3856,75.13,86.77,...,80.86,69.31,74.35,3450.00,85.56,80.90,82.91,3190.00,None,NaN


In [25]:
# Close the Log File
log_file.close()
sys.stdout = sys.__stdout__

# Compare between different classifiers' accuracies

In [26]:
result_path_bca= path + "/BCA"
result_path_ar = path + "/CAR"

In [27]:



# Load Correlation DataFrames from BCA folders
df_bin_clf_bca = pd.read_csv(result_path_bca+'/bin_clf_result.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_mcl_clf_bca = pd.read_csv(result_path_bca+'/mcl_clf_result.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_bin_clf_ar = pd.read_csv(result_path_ar+'/bin_clf_result.csv').drop("Unnamed: 0", axis=1, errors="ignore")
df_mcl_clf_ar = pd.read_csv(result_path_ar+'/mcl_clf_result.csv').drop("Unnamed: 0", axis=1, errors="ignore")


df_bin_clf_bca['Number of Classes'] = 2
df_mcl_clf_bca['Number of Classes'] = 3
df_bin_clf_ar['Number of Classes'] = 2
df_mcl_clf_ar['Number of Classes'] = 3

df_bin_clf_bca['Pre-Processed Algorithm'] = 'Baseline Correction Method' 
df_mcl_clf_bca['Pre-Processed Algorithm'] = 'Baseline Correction Method'
df_bin_clf_ar['Pre-Processed Algorithm'] = 'Average Common Referencing' 
df_mcl_clf_ar['Pre-Processed Algorithm'] = 'Average Common Referencing' 

df_concatenate = pd.concat([df_bin_clf_bca, df_mcl_clf_bca, df_bin_clf_ar, df_mcl_clf_ar ], ignore_index=True)
df_concatenate.to_csv(path+'/all_result.csv')
df_concatenate

,Focus,Data,FrequencyBand,X_shape,y_shape,Classifier,CV,Accuracy,Expert Precision,Expert Recall,...,Novice F1-Score,Novice Support,Best Group,Best Accuracy,Number of Classes,Pre-Processed Algorithm,Intermediate Precision,Intermediate Recall,Intermediate F1-Score,Intermediate Support
0,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Stratified10-fold CV,85.9475,88.08,86.73,...,84.22,3190.00,NaN,NaN,2,Baseline Correction Method,NaN,NaN,NaN,NaN
1,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Subject-Out CV,76.6333,63.16,57.24,...,36.61,1329.17,[38],100.0000,2,Baseline Correction Method,NaN,NaN,NaN,NaN
2,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",RF,Leave-One-Algorithm-Out CV,86.4624,88.15,87.84,...,84.59,996.88,['HeightOfTree'],95.8333,2,Baseline Correction Method,NaN,NaN,NaN,NaN
3,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Stratified10-fold CV,83.8984,80.81,93.87,...,79.29,3190.00,NaN,NaN,2,Baseline Correction Method,NaN,NaN,NaN,NaN
4,Algorithm,CodeComprehension,4to50hz,"(727, 1)","(727,)",SVM,Leave-One-Subject-Out CV,83.7500,75.00,74.61,...,64.29,2278.57,[71],100.0000,2,Baseline Correction Method,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Subject-Out CV,58.3615,50.00,47.46,...,35.19,4042.11,[71],100.0000,3,Average Common Referencing,34.78,17.60,20.56,3339.13
212,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",SVM,Leave-One-Electrode-Out CV,66.9342,63.09,89.06,...,66.43,1200.00,['C6'],75.6757,3,Average Common Referencing,67.07,50.00,56.96,1200.00
213,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Stratified10-fold CV,79.0964,72.49,81.01,...,86.09,7680.00,NaN,NaN,3,Average Common Referencing,78.83,71.75,75.05,7680.00
214,ElectrodePosition,CodeComprehension,AlphaBetaThetaGamma,"(2368, 4)","(2368,)",XGB,Leave-One-Subject-Out CV,55.1098,52.17,39.61,...,21.87,2400.00,[24],100.0000,3,Average Common Referencing,33.33,14.77,17.76,2327.27
